tokenizer.model : 바이너리 파일, (모델 파라미터 + vocab 포함)

import sentencepiece as spm; spm.SentencePieceProcessor(model_file="tokenizer.model")

tokenizer.json : fast 토크나이저 전용 JSON, vocab, merege table, 특수토큰, normalizer 설정이 모두 있음
AutoTokenizer.from_pretrained(tokenizer_file="tokenizer.json")

special_tokens_map.json : <unk>, <pad>, <eos> : 특수 토큰

added_tokens.json : 기본 vocab 이후 사용자가 tokenizer.add_tokens() 추가한 토큰 목록만 따로 저장.

vocab.json : BPE, Wordpiece 계열이면 vocab.json(토큰=>ID), merges.txt(merge 규칙) 상이 생김. SentencePiece에는 해당 안됨

In [ ]:
# pipeline을 쓰면 기본 모델을 로드해서 사용함

In [ ]:
from transformers import pipeline
qa = pipeline("question-answering")   # ← 기본 모델 자동 선택
print(type(qa.model))                # <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForQuestionAnswering'>
print(qa.model.name_or_path)         # distilbert-base-cased-distilled-squad
print(type(qa.tokenizer))            # <class 'transformers.models.bert.tokenization_bert_fast.BertTokenizerFast'>


In [ ]:
# 세부적으로 지정할 수 있음

qa = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",   # 원하는 체크포인트
    tokenizer="deepset/roberta-base-squad2",
    revision="refs/convert/parquet-2025-03-22"  # 고정된 revision 예시
)


![Image](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline-dark.svg)

In [ ]:
from_pretraiend
device_map를 통해 매핑표를 전달

"auto", "balanced", …	: Accelerate가 알아서 분산(레이어 스플릿)
{"": "cuda:0"} : 루트 모듈 전체를 단일 GPU 0에 둬라
{"model.embed_tokens":0, "model.layers.0":0, "model.layers.1":1} : 서브모듈별 수동 배치

# 서브모듈별로 조절가능
device_map = {
    "vision_model":   "cuda:0",   # 이미지 타워
    "language_model": "cuda:1",   # 텍스트 타워
    "":               "cpu"       # 나머지(혹은 디스크)
}
